# Агент помощник оператора

Локальная LLM выбирает инструменты, сервер проверяет политику и возвращает ответ с цитатами, уточняющий вопрос или черновик передачи оператору. Черновик не отправляется и не сохраняется как реальный тикет. Агент использует существующий поисковый API с его откалиброванным BM25; dense/RRF остаются отдельным исследовательским контуром до калибровки отказов.


In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
report_path = ROOT / 'reports/agent_validation.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('Успешных demo-сценариев:', report['passed'], '/', report['total'])
    display(pd.DataFrame(report['cases']))
else:
    print('Сначала выполните scripts/verify_agent_http.py при запущенном API.')


## Запуск сервера

В терминале задайте SUPPORT_API_KEY и выполните `docker compose -f compose.yaml -f compose.llm.yaml up --build -d --wait`. Модель qwen2.5:0.5b должна быть загружена в Ollama. POST /v1/agent/run принимает text, language и max_steps от 1 до 6. Ключ передаётся заголовком x-api-key. Контекст одного запроса не хранится между вызовами; для уточнения отправьте новый текст с нужными деталями.


In [ ]:
RUN_LIVE_REQUEST = False
API_ROOT = 'http://127.0.0.1:8000'
payload = {'text': 'Как установить зависимости проекта?', 'language': 'ru', 'max_steps': 4}
if RUN_LIVE_REQUEST:
    from urllib.request import Request, urlopen
    headers = {'Content-Type': 'application/json', 'x-api-key': os.environ.get('SUPPORT_API_KEY', '')}
    request = Request(API_ROOT + '/v1/agent/run', data=json.dumps(payload).encode(), headers=headers, method='POST')
    with urlopen(request, timeout=90) as response:
        result = json.load(response)
    print('Статус:', result['status'], '| Причина:', result['reason'])
    print('Ответ:', result['answer'])
    print('Уточнение:', result['clarification'])
    display(pd.DataFrame(result['trace']))
    display(result['citations'])
    display(result['ticket_draft'])
else:
    print('Для собственного запроса задайте RUN_LIVE_REQUEST=True при запущенном сервере.')


## Что разбирать

Посмотрите последовательность tool в trace. Уточнение и черновик — терминальные результаты. prepare_answer запрещён до успешного поиска с threshold_passed. Повтор инструмента, неизвестный tool, сбой провайдера и исчерпание бюджета приводят к безопасной передаче. Это не означает успешного решения задачи: корректный ответ, правильное уточнение и обоснованная передача оцениваются отдельно. Trace содержит действия и время шагов, а не внутренние рассуждения модели.
